# Previsão de Risco de Crédito

## Fase 01 — Análise Exploratória de Dados (EDA)

### Contexto do problema

Uma instituição financeira precisa avaliar o risco associado à concessão
de empréstimos. Neste projeto, será desenvolvido um modelo de Machine Learning
capaz de auxiliar na identificação de clientes com maior risco de inadimplência.

A variável alvo é `loan_status`:

- `0`: cliente não inadimplente;
- `1`: cliente inadimplente.

### Objetivo desta etapa

A Análise Exploratória de Dados tem como objetivo compreender a estrutura,
a qualidade e o comportamento da base antes de qualquer tratamento ou
modelagem.

Nesta etapa serão investigados:

- estrutura e tipos das variáveis;
- estatísticas descritivas;
- valores ausentes e registros duplicados;
- distribuição das variáveis;
- distribuição da variável alvo;
- possíveis valores discrepantes;
- relações e correlações entre as variáveis.

In [37]:
#Importantdo bibliotecas para análise de dados e visualização
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [38]:
#Lendo o arquivo CSV contendo o dataset de risco de crédito
df = pd.read_csv('../data/credit_risk_dataset.csv')

#visualizando as primeiras linhas do dataset
display(df.head())

#conferindo o tamanho do dataset
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


O dataset possui 32581 linhas e 12 colunas.


In [39]:
#Explorando o dataset para entender melhor suas características e estrutura
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


In [40]:
#Descrobindo estatísticas descritivas do dataset
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [41]:
#Explorando as colunas do tipo string para entender melhor suas características
df.describe(include="str")

,person_home_ownership,loan_intent,loan_grade,cb_person_default_on_file
count,32581,32581,32581,32581
unique,4,6,7,2
top,RENT,EDUCATION,A,N
freq,16446,6453,10777,26836


In [42]:
#Somandos os valores nulos em cada coluna do dataset
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [43]:
dados_ausentes = pd.DataFrame(
    {"Quantidade": df.isnull().sum(),
     "Percentual (%)": (df.isnull().sum() / df.shape[0]) * 100}
)

dados_ausentes[dados_ausentes["Quantidade"] > 0]

,Quantidade,Percentual (%)
person_emp_length,895,2.747000
loan_int_rate,3116,9.563856


In [44]:
#explorando as duplicatas no dataset
df.duplicated().sum()

np.int64(165)

In [47]:
#Explorando a coluna 'loan_status' para entender melhor a distribuição dos valores
df['loan_status'].value_counts()
df['loan_status'].value_counts(normalize=True) * 100

loan_status
0    78.183604
1    21.816396
Name: proportion, dtype: float64

## Primeiras observações da base

A base possui 32.581 registros e 12 variáveis, contendo atributos
numéricos e categóricos relacionados ao perfil dos clientes e às
características dos empréstimos.

Foram identificados valores ausentes em duas variáveis:
`person_emp_length`, com 895 registros ausentes, e `loan_int_rate`,
com 3.116 registros ausentes. A técnica de tratamento desses valores
será definida posteriormente a partir da análise das distribuições.

A estatística descritiva também revelou possíveis valores discrepantes.
A variável `person_age`, por exemplo, apresenta valor máximo de 144 anos,
enquanto 75% dos registros possuem idade de até 30 anos. De forma
semelhante, `person_emp_length` apresenta máximo de 123 anos de vínculo,
apesar de 75% dos valores estarem abaixo de 7 anos. Esses registros serão
investigados por meio de análises gráficas antes da definição de qualquer
tratamento.

A variável `person_income` também apresenta forte amplitude, com mediana
de 55.000 e valor máximo de 6.000.000, sugerindo uma possível distribuição
assimétrica e a presença de valores extremos.

A variável alvo `loan_status` apresenta média aproximada de 0,218. Como
se trata de uma variável binária, esse resultado indica preliminarmente
que cerca de 21,8% das observações pertencem à classe de inadimplência,
sugerindo desbalanceamento entre as classes. Essa distribuição será
investigada diretamente nas próximas etapas da EDA.

Também foram identificados 165 registros duplicados. Nesta etapa nenhuma
remoção, imputação ou transformação foi realizada, pois o objetivo da EDA
é primeiro compreender os problemas existentes antes da definição das
estratégias de tratamento.